## Analyze Product Token Lengths for Fixed Chunking

This cell measures the token length of each product document using the existing `count_tokens()` function.  
The goal is to understand the distribution of document sizes and choose an appropriate fixed chunk size.

We will:
- count tokens for every product document
- calculate summary statistics: `p50`, `p75`, `p90`, `p95`, and `max`
- inspect how large most product records are
- choose a fixed chunk size slightly above the percentile that keeps most products in a single chunk

This helps ensure that product information stays together as one chunk whenever possible, which is usually better for product retrieval.

In [1]:
from pathlib import Path
import math
import sys

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing 'src'")

In [3]:
def percentile(values: list[int], p: int) -> float:
    if not values:
        raise ValueError("Cannot compute percentiles for an empty list")
    if len(values) == 1:
        return float(values[0])

    values = sorted(values)
    rank = (len(values) - 1) * (p / 100)
    lower = math.floor(rank)
    upper = math.ceil(rank)

    if lower == upper:
        return float(values[int(rank)])

    weight = rank - lower
    return values[lower] * (1 - weight) + values[upper] * weight

In [4]:
def round_up(value: float, base: int = 50) -> int:
    return int(math.ceil(value / base) * base)

In [5]:
PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from services.ingest_services import count_tokens, load_jsonl_docs, load_markdown_docs


SOURCE = "jsonl"  # change to "markdown" to analyze markdown docs instead
BUFFER_MULTIPLIER = 1.05  # 5% above the selected percentile
ROUND_TO = 50
JSONL_PATH = PROJECT_ROOT / "data" / "kapruka_docs.jsonl"

loader = load_jsonl_docs if SOURCE == "jsonl" else load_markdown_docs
docs = loader(JSONL_PATH if SOURCE == "jsonl" else None)

2026-05-24 19:37:08.186 | INFO     | services.ingest_services.pipeline:load_jsonl_docs:175 - Loaded 96 documents from JSONL in e:\Tutorials\Zuu\AI Engineer Essentials\Labs\Mini Project 3\kapruka_agent\data\kapruka_docs.jsonl


In [6]:
rows = []
for doc in docs:
    tokens = count_tokens(doc["content"])
    rows.append(
        {
            "title": doc.get("product_name") or doc.get("title") or "Untitled",
            "product_id": doc.get("product_id", ""),
            "url": doc.get("url", ""),
            "tokens": tokens,
        }
    )

In [7]:
token_counts = [row["tokens"] for row in rows]

stats = {
    "documents": len(token_counts),
    "p50": percentile(token_counts, 50),
    "p75": percentile(token_counts, 75),
    "p90": percentile(token_counts, 90),
    "p95": percentile(token_counts, 95),
    "max": max(token_counts),
}

In [8]:
recommended_chunk_size = round_up(stats["p95"] * BUFFER_MULTIPLIER, ROUND_TO)
single_chunk_docs = sum(tokens <= recommended_chunk_size for tokens in token_counts)
coverage = single_chunk_docs / len(token_counts)

In [9]:
print(f"Source: {SOURCE}")
print(f"Documents analyzed: {stats['documents']}")
print("\nToken length percentiles:")
for key in ["p50", "p75", "p90", "p95", "max"]:
    print(f"  {key}: {stats[key]:.1f}")

Source: jsonl
Documents analyzed: 96

Token length percentiles:
  p50: 181.0
  p75: 209.8
  p90: 244.0
  p95: 259.0
  max: 291.0


In [10]:
print("\nRecommended fixed chunk size:")
print(
    f"  {recommended_chunk_size} tokens "
    f"(rounded up from p95 * {BUFFER_MULTIPLIER:.2f})"
)
print(f"  Coverage at this size: {single_chunk_docs}/{len(token_counts)} ({coverage:.1%})")


Recommended fixed chunk size:
  300 tokens (rounded up from p95 * 1.05)
  Coverage at this size: 96/96 (100.0%)


In [11]:
print("\nTop 10 longest product documents:")
for row in sorted(rows, key=lambda item: item["tokens"], reverse=True)[:10]:
    print(f"  {row['tokens']:>5} tokens | {row['product_id']} | {row['title']}")


Top 10 longest product documents:
    291 tokens | ef_pc_elec0v18pod00255p | Usha Rc 280a Multi Cooker Rice Cooker
    286 tokens | ef_pc_elec0v18pod00040p | AMILEX 1.8L Rice Cooker With Steamer
    284 tokens | ef_pc_elec0v18pod00041p | AMILEX 2.8L Rice Cooker With Steamer
    273 tokens | ef_pc_elec0v3851pod00052p | BLACK And DECKER 0.6L Rice Cooker With Glass Lid RC650B5
    262 tokens | ef_pc_elec0v18pod00261p | Richpower Rprc 6086 Electric Rice Cooker
    258 tokens | ef_pc_elec0v18pod00331p | Infinity Electric Rice Cooker
    255 tokens | ef_pc_elec0v18pod00263p | Richpower Rprc 6087 Electric Rice Cooker
    250 tokens | ef_pc_elec0v18pod00274p | Mitshu Mrc Cb 10 Electric Rice Cooker
    245 tokens | ef_pc_elec0v3851pod00053p | BLACK And DECKER 1.0L Nonstick Rice Cooker With Glass Lid RC1050B5
    245 tokens | ef_pc_elec0v3851pod00053p | BLACK And DECKER 1.0L Nonstick Rice Cooker With Glass Lid RC1050B5
